# AstraDB, CassandraDB

### Apache Cassandra-
- open-source NoSQL distributed database
- designed for high availability,scalability,massive data storage

### AstraDB-
- managed cassandra in the cloud built by Data Stax
- vector search,automatic embedding storage

In [19]:
import os
from dotenv import load_dotenv
load_dotenv()
astra_db_api=os.getenv("ASTRA_DB_API_ENDPOINT")
astra_db_token=os.getenv("ASTRA_DB_APPLICATION_TOKEN")
astra_db_keyspace=os.getenv("ASTRA_DB_KEYSPACE")
os.environ["HUGGINGFACE_TOKEN"]=os.getenv("HUGGINGFACE_TOKEN")
groq_api_key=os.getenv("GROQ_API_KEY")

In [10]:
from langchain_astradb import AstraDBVectorStore
#AstraDBVectorStore- automatically creates the embedding collection,vector column

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
loader=PyPDFLoader("attention.pdf")
docs=loader.load()

In [14]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=50
)
chunks=splitter.split_documents(docs)

In [15]:
embeddings=HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

In [16]:
#astradb vectorstore
vector_store=AstraDBVectorStore(
    embedding=embeddings,
    collection_name="pdf_query",
    api_endpoint=astra_db_api,
    token=astra_db_token,
    namespace=astra_db_keyspace
)
#store embeddings
vector_store.add_documents(chunks)

['2dc6788453cb472b85019302903a0802',
 '1cff9f4285bc4759994b91e88f1a4ac2',
 '72ea960484b644dc86007e83510c9827',
 'b9f9ff16091a46c7a6b83afa36f6a270',
 '4b148c9cd54a470e9f9bf2492aa2c1cf',
 'dfb8075ae8174f6893e4784b024505de',
 '150f2a007165415b9d3a11042d724414',
 'a532082948014a68acaa1b45849eafc2',
 'a223fa63d4f7407c940c3494f6d30565',
 '7186e6c864284742ad82f48c7cb9e27e',
 'e03f4fbeb0434836b52e32a11f5c2788',
 '56f1ce3e76ea42eeaf9015535d4c1b72',
 '1932cecf33834ca4b82aaf9dda972bab',
 'ed65921aa1654519be3e1416d33cd491',
 'bac300d509214eeab780f08a8a47491c',
 'd084a3e2e9344774861bc0e8aed4f252',
 '55eecdf8202b4345b0f8c6d8c69f600b',
 '7534784b145d4f63929487daf260ace3',
 'c46d8a6fc53f4d69bdd318e7351ee910',
 '32acca203dd24f60bb805bbcfcaf611f',
 '85879dde88f946ec8748864b8c9954cb',
 'f6e6d84b15d84701a6d87d6bcd7bf582',
 '0907cf4e551640cbb1635e503cc79bef',
 '5dddda1ed24a4003a221ddfd04562a13',
 '655754b1554a45dfab7c0dd6a1f8dc6f',
 '251b702fa521495fa13c366d38f4a1c1',
 '9d3b4bfbec7a436dae8b70dbf3386a86',
 

In [17]:
#create retriever
retriever=vector_store.as_retriever(search_top_k=3)

In [ ]:
#Ask question
query="what is self attention?"
document=retriever.invoke(query)
print(document)

[Document(id='655754b1554a45dfab7c0dd6a1f8dc6f', metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing b

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=groq_api_key
)
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided.
Answer the question in 3 lines maximum.

Context:
{context}

Question:
{question}
""")

context="\n\n".join([d.page_content for d in document])

chain=prompt|llm

result=chain.invoke({
    "context":context,
    "question":query
})
print(result.content)

Self-attention refers to a mechanism where a position in a sequence can attend to all other positions in the same sequence.
This allows each position to weigh and combine information from other positions based on their similarity.
It's a key component in models like transformers where positions can attend to all positions in the sequence.
